# Module 14: How Big an Effect Could You Have Detected?

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Before an evaluation starts, and before a null result is believed, one number
should be on the table: **the smallest effect this design had a fair chance of
finding.**

It takes four lines to compute and it changes how every other number is read.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
prof = profile.set_index("agency_id")

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated_ids, outcome="n_uof", offset=None):
    """The standard specification, with whoever is labelled treated."""
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated_ids))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated_ids))
                  & (s["period"] == "phase")).astype(float)
    off = s["lo"] if offset is None else offset
    z = smf.glm(f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson(), offset=off).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z.bse["settled"]

## 2. The minimum detectable effect

The rule of thumb: with 80 percent power and a 5 percent two sided test, an
effect has to be about **2.8 standard errors** from zero to be found reliably.

The standard error is the one the model already reports.

In [ ]:
e, lo, hi, se = fit(d, keep)
mde = 100 * (1 - np.exp(-2.80 * se))
print(f"  standard error on the log scale:  {se:.4f}")
print(f"  smallest detectable effect:       {mde:.1f} percent")
print(f"\n  the study's estimate:             {e:+.1f} percent")
print(f"  the truth:                        {TRUTH:+.1f} percent")

The design could reliably find an effect of **8.6 percent or larger**, and the
real effect is 12. That is why this study worked.

**Turn that sentence around and it is the one every null result needs.** A
study with a detectable effect of 20 percent that reports "no significant
change" has reported how small its sample was.

## 3. What the detectable effect depends on

In [ ]:
rows = []
for label, end in [("8 months", "2024-06"), ("14 months", "2024-12"),
                   ("20 months", "2025-06"), ("26 months", "2025-12"),
                   ("30 months", "2026-04")]:
    s = d[d["year_month"] <= end]
    _, _, _, se2 = fit(s, keep)
    rows.append({"follow up": label,
                 "smallest detectable effect": f"{100 * (1 - np.exp(-2.80 * se2)):.1f}%",
                 "can it see 12 percent": "yes" if 100 * (1 - np.exp(-2.80 * se2)) < 12 else "NO"})
pd.DataFrame(rows).set_index("follow up")

In [ ]:
rows = []
for k in [1, 2, 3, 4]:
    _, _, _, se3 = fit(d, keep[:k])
    rows.append({"agencies given the program": k,
                 "smallest detectable effect": f"{100 * (1 - np.exp(-2.80 * se3)):.1f}%"})
pd.DataFrame(rows).set_index("agencies given the program")

Follow up length matters a great deal: 14.2 percent at eight months, 8.6 at
thirty.

**The number of treated agencies barely matters at all.** Going from one to
four buys about one percentage point. That is not what most people expect,
and the reason is worth understanding: precision depends on the incidents on
**both** sides of the comparison, and the comparison group here is dominated
by a single large agency whose contribution is fixed.

**Adding treated units to a study whose comparison group is thin is close to
useless.** Before designing a rollout, compute this table.

## 4. Reporting a null result honestly

| Instead of | Write |
|---|---|
| "no significant effect" | "no effect was detected; the study could detect a reduction of 8.6 percent or more" |
| "the program did not work" | "an effect smaller than 8.6 percent would not have been visible here" |
| "results were inconclusive" | "the interval runs from 17.9 percent below to 6.9 percent above" |

The interval does most of this work by itself, which is the argument for
always printing it. The detectable effect adds the piece the interval leaves
implicit: what the study was **built** to find, rather than what it happened
to find.

## Exercise

Suppose the program had a true effect of 5 percent rather than 12. Simulate
the study and see how often it would have been detected.

In [ ]:
# Fill in the blank, then run.
TRUE_EFFECT = None          # try 5.0, then 12.0

if TRUE_EFFECT is not None:
    rng = np.random.default_rng(15)
    hits = 0
    reps = 150
    for _ in range(reps):
        s = d.copy()
        mult = np.where((s["agency_id"].isin(keep)) & (s["period"] == "after"),
                        1 - TRUE_EFFECT / 100, 1.0)
        s["y"] = rng.poisson(s["n_uof"].values * mult)
        _, lo2, hi2, _ = fit(s, keep, outcome="y")
        hits += not (lo2 < 0 < hi2)
    print(f"  a true effect of {TRUE_EFFECT:.0f} percent is detected "
          f"{100 * hits / reps:.0f} percent of the time")
    print(f"  the design's stated detectable effect is "
          f"{100 * (1 - np.exp(-2.80 * se)):.1f} percent")
else:
    print("Set TRUE_EFFECT above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

Run it twice.

```python
TRUE_EFFECT = 5.0     # then 12.0
```

A 12 percent effect is found nearly every time, comfortably above the 80
percent the calculation promises. A 5 percent effect is found far less
often, which is what a detectable effect of 8.6 percent means.

**The simulation is the honest version of the rule of thumb.** The 2.8
standard errors calculation assumes the standard error does not change when
the effect does, which is not exactly true for count data, and it ignores
that the estimate has to clear the threshold in the right direction.

Running 150 simulations takes under a minute and gives the actual answer for
the actual design. When a funder asks whether a study can detect the effect
they hope for, this is the cell to run, before the study starts rather than
after it fails.

</details>

---

**Next:** [Module 15: Sensitivity, How Wrong Would the Assumption Have to Be?](Module_15_Sensitivity.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*